# 08 Example days: a conformant and a non-conformant Volt-VAr site

In [ ]:
import sys
from pathlib import Path

# Put the repo root on sys.path 
_current = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import duckdb
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

from bms_sa_review.ami_analysis.lib import ami_fleet_vvar as Fleet
from shared.ciccada_config import AS4777

## 1. Settings

In [ ]:
# Same method settings as notebook 06 (main scenario)
MAIN = dict(v_lo=242, v_hi=253, p_min_frac=Fleet.P_MIN_FRAC, night_hours=(0, 4))

# Site-month candidates: clear examples only
MIN_N_WINDOW = 300       # fitted intervals required in BOTH sources
MIN_R2_CONFORM = 0.8     # conformant example: clean fit in both sources
MAX_FLAT_SLOPE = 0.3     # non-conformant example: |slope| below this in both sources (%/V)

# Figure
DAY_HOURS = (0, 24)                         # x-axis range (AEST hours)
P_MIN_PCT = AS4777["QCAP"]["P_MIN"] * 100   # 20% of rating: below this, no Q requirement
TOL_PCT = AS4777["TOL_FRAC"] * 100          # +/- band around the required Q (% of rating)
AEST = pd.Timedelta(hours=10)               # store timestamps are UTC; AEST = UTC+10

# Inputs / outputs
RESULTS_DIR = Fleet.store_dir() / "vvar_validation" / "all_sites" / "band_242_253"
THREE_WAY = RESULTS_DIR / "three_way_site_month.csv"
FIG_DIR = RESULTS_DIR / "paper_figures"
FIG_DIR.mkdir(exist_ok=True)
print(f"Reading {THREE_WAY}")

## 2. Site metadata (rating)

In [ ]:
con = duckdb.connect()
site_meta = con.sql(f"""
    SELECT site_id, ac_capacity_kw AS rating_kw
    FROM read_parquet('{Fleet.Config.store_path("ami_site_metadata_residential").as_posix()}')
""").df()
con.close()
rating = site_meta.set_index("site_id").rating_kw

## 3. Candidate site-months

In [ ]:
tw = pd.read_csv(THREE_WAY)
both = tw.dropna(subset=["slope_blind", "slope_truth"]).copy()
for c in ["pass_blind", "pass_truth"]:
    both[c] = both[c].astype(str).eq("True")      # CSV stores booleans as text
both = both.loc[(both.n_window_blind >= MIN_N_WINDOW) & (both.n_window_truth >= MIN_N_WINDOW)]
both["month_str"] = both.year.astype(int).astype(str) + "-" + both.month.astype(int).map("{:02d}".format)
both["abs_diff"] = (both.slope_blind - both.slope_truth).abs()

COLS = ["site_id", "month_str", "slope_blind", "slope_truth", "r_squared_blind", "r_squared_truth",
        "n_window_blind", "n_window_truth"]

conform = (both.loc[both.pass_blind & both.pass_truth
                    & (both.r_squared_blind >= MIN_R2_CONFORM) & (both.r_squared_truth >= MIN_R2_CONFORM)]
               .sort_values(["abs_diff", "n_window_truth"], ascending=[True, False]))
nonconform = (both.loc[~both.pass_blind & ~both.pass_truth
                       & (both.slope_blind.abs() < MAX_FLAT_SLOPE) & (both.slope_truth.abs() < MAX_FLAT_SLOPE)
                       & (both.slope_truth != 0)]
                  .sort_values("n_window_truth", ascending=False))

print(f"{len(conform):,} conformant and {len(nonconform):,} non-conformant candidate site-months")
display(conform[COLS].head(10).round(3))
display(nonconform[COLS].head(10).round(3))

In [ ]:
# Pick one row from each table above (position in the sorted table)
CONFORM_PICK = 0
NONCONF_PICK = 1

picks = {"conformant": conform.iloc[CONFORM_PICK], "non-conformant": nonconform.iloc[NONCONF_PICK]}
for label, r in picks.items():
    print(f"{label:15s} site {int(r.site_id)}  {r.month_str}  "
          f"slope unmonitored {r.slope_blind:.2f}, reference {r.slope_truth:.2f} %/V")

## 4. Choose the day


In [ ]:
SOURCES = {"reference": "ground_truth", "unmonitored": "meter_baseline"}

def month_points(site_id, month, source):
    """All intervals of one site-month for one source, with AEST time and % of rating."""
    site = pd.DataFrame({"site_id": [site_id], "rating_kw": [rating[site_id]]})
    pts = Fleet.fetch_site_points(month, site, source, **MAIN)
    t = pd.to_datetime(pts.t_stamp)
    if t.dt.tz is not None:
        t = t.dt.tz_convert("UTC").dt.tz_localize(None)
    pts["t"] = t + AEST
    pts["date"] = pts.t.dt.date
    pts["p_pct"] = 100.0 * pts.p_est / rating[site_id]
    return pts

data = {}
for label, r in picks.items():
    data[label] = {name: month_points(int(r.site_id), r.month_str, src) for name, src in SOURCES.items()}
    ranked_days = (data[label]["reference"].loc[lambda d: d.in_window]
                   .groupby("date").size().sort_values(ascending=False))
    print(f"\n{label}: days with most reference intervals in the window")
    print(ranked_days.head(5).to_string())

In [ ]:
DAY_OVERRIDE = {}     # e.g. {"conformant": "2024-01-14", "non-conformant": "2024-11-03"}

days = {}
for label in picks:
    ref = data[label]["reference"]
    top = ref.loc[ref.in_window].groupby("date").size().idxmax()
    days[label] = pd.Timestamp(DAY_OVERRIDE.get(label, top)).date()
    print(f"{label:15s} -> {days[label]}")

## 5. Figure

In [ ]:
C_REF, C_EST, C_REQ, C_V = "#eb6834", "#2a78d6", "#f59e0b", "#b45309"
C_GREY, C_WIN, C_GRID = "#d1d5db", "#7c3aed", "#ebebeb"
VV = AS4777["VVAR"]

def required_q_pct(v):
    """AS/NZS 4777.2:2020 Australia A Volt-VAr curve, % of rating (negative = absorbing)."""
    return np.interp(v, [VV["V1"], VV["V2"], VV["V3"], VV["V4"]],
                     [VV["Q1"] * 100, 0.0, 0.0, -VV["Q4"] * 100])

def plot_day_pair(columns, figsize=(11, 8)):
    """
    columns: list of (title, ref_day, est_day). One column per site; rows = V, P, Q.
    Greyed: PV-circuit output below P_MIN_PCT (no Q requirement).
    """
    fig, axes = plt.subplots(3, len(columns), figsize=figsize, dpi=300, sharex="col", sharey="row",
                             layout="constrained", gridspec_kw={"height_ratios": [1, 1, 1.4]})
    for j, (title, ref, est) in enumerate(columns):
        ax_v, ax_p, ax_q = axes[:, j]
        low = (ref.p_pct.abs() < P_MIN_PCT).to_numpy()

        for ax in (ax_v, ax_p, ax_q):
            ax.fill_between(ref.t, 0, 1, where=low, transform=ax.get_xaxis_transform(),
                            color=C_GREY, alpha=0.6, linewidth=0, step="mid", zorder=0)
            ax.grid(color=C_GRID, linewidth=0.5)

        # Voltage, with the fitted window shaded
        ax_v.axhspan(MAIN["v_lo"], MAIN["v_hi"], color=C_WIN, alpha=0.08, linewidth=0)
        ax_v.plot(ref.t, ref.V, color=C_V, lw=1.1)
        for v, ls in [(VV["V3"], ":"), (MAIN["v_lo"], "--"), (MAIN["v_hi"], "--")]:
            ax_v.axhline(v, color=C_V, lw=0.7, ls=ls, alpha=0.7)

        # Active power
        ax_p.plot(ref.t, ref.p_pct, color=C_REF, lw=1.1)
        ax_p.plot(est.t, est.p_pct, color=C_EST, lw=1.0, alpha=0.85)
        ax_p.axhline(P_MIN_PCT, color="black", lw=0.7, ls=":")

        # Reactive power vs the required curve (+/- tolerance), using the measured voltage
        req = required_q_pct(ref.V.to_numpy())
        ax_q.fill_between(ref.t, req - TOL_PCT, req + TOL_PCT, color=C_REQ, alpha=0.25, linewidth=0)
        ax_q.plot(ref.t, req, color=C_REQ, lw=1.0)
        ax_q.plot(ref.t, ref.q_pct, color=C_REF, lw=1.1)
        ax_q.plot(est.t, est.q_pct, color=C_EST, lw=1.0, alpha=0.85)
        ax_q.axhline(0, color="black", lw=0.5)

        ax_v.set_title(title, fontsize=9, fontweight="bold")
        day = ref.t.dt.normalize().iloc[0]
        ax_q.set_xlim(day + pd.Timedelta(hours=DAY_HOURS[0]), day + pd.Timedelta(hours=DAY_HOURS[1]))
        ax_q.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
        ax_q.xaxis.set_major_locator(mdates.HourLocator(interval=3))
        ax_q.set_xlabel("Time (AEST)", fontsize=8)

    axes[0, 0].set_ylabel("Voltage (V)", fontsize=8)
    axes[1, 0].set_ylabel("Active power\n(% of rating)", fontsize=8)
    axes[2, 0].set_ylabel("Reactive power (% of rating)\n+ supplying, − absorbing", fontsize=8)
    for ax in axes.flat:
        ax.tick_params(labelsize=7)

    handles = [
        plt.Line2D([], [], color=C_REF, lw=1.2, label="PV-circuit reference"),
        plt.Line2D([], [], color=C_EST, lw=1.2, label="Unmonitored estimate (net meter − monthly night baseline)"),
        Patch(color=C_REQ, alpha=0.4, label=f"Required Q (±{TOL_PCT:.0f}% of rating)"),
        Patch(color=C_WIN, alpha=0.2, label=f"Fitted window ({MAIN['v_lo']}–{MAIN['v_hi']} V)"),
        Patch(color=C_GREY, alpha=0.8, label=f"Output < {P_MIN_PCT:.0f}% of rating (no Q requirement)"),
    ]
    fig.legend(handles=handles, loc="outside lower center", ncol=3, fontsize=7.5, frameon=False)
    return fig

In [ ]:
columns = []
for letter, (label, r) in zip("ab", picks.items()):
    ref, est = (data[label][k].loc[lambda d: d.date == days[label]].sort_values("t")
                for k in ("reference", "unmonitored"))
    title = (f"({letter}) {label.capitalize()} site, {days[label]}\n"
             f"site-month slope: {r.slope_truth:.2f} %/V reference, {r.slope_blind:.2f} %/V unmonitored")
    columns.append((title, ref, est))

fig = plot_day_pair(columns)
fig.savefig(FIG_DIR / "example_days_conformant_vs_not.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "example_days_conformant_vs_not.png", bbox_inches="tight")
plt.show()
print(f"Saved to {FIG_DIR}")

## 6. Single day single site

In [ ]:
# ---------------------------------------------------------------------------
# Single-site figure: pick any site and day (same rows and styling as above)
# ---------------------------------------------------------------------------
SITE_ID = 1584960641          # any site in the cohort
DAY = "2025-12-15"            # AEST date to plot

day = pd.Timestamp(DAY)
month = day.strftime("%Y-%m")
if SITE_ID not in rating.index:
    raise ValueError(f"Site {SITE_ID} not found in the site metadata.")

# All intervals of that month for both sources, then keep the chosen day
pts = {name: month_points(SITE_ID, month, src) for name, src in SOURCES.items()}
ref, est = (pts[k].loc[lambda d: d.date == day.date()].sort_values("t") for k in ("reference", "unmonitored"))
if ref.empty:
    days_avail = sorted(pts["reference"].date.unique())
    raise ValueError(f"No reference data for site {SITE_ID} on {DAY}. "
                     f"Days available in {month}: {days_avail[:5]} ... {days_avail[-5:]}")

# Site-month slopes and verdicts from notebook 06, if this site-month was assessable by both methods
row = both.loc[(both.site_id == SITE_ID) & (both.month_str == month)] if "both" in globals() else pd.DataFrame()
if not row.empty:
    r = row.iloc[0]
    verdict = lambda p: "conformant" if p else "non-conformant"
    subtitle = (f"site-month slope: {r.slope_truth:.2f} %/V reference ({verdict(r.pass_truth)}), "
                f"{r.slope_blind:.2f} %/V unmonitored ({verdict(r.pass_blind)})")
else:
    subtitle = "site-month not assessable by both methods (or not in the candidate table)"
title = f"Site {SITE_ID}, {DAY}\n{subtitle}"


def plot_day_single(title, ref, est, figsize=(6, 8)):
    """One site-day: rows = voltage, active power, reactive power. Greyed: PV output < P_MIN_PCT."""
    fig, (ax_v, ax_p, ax_q) = plt.subplots(3, 1, figsize=figsize, dpi=300, sharex=True,
                                           layout="constrained", gridspec_kw={"height_ratios": [1, 1, 1.4]})
    low = (ref.p_pct.abs() < P_MIN_PCT).to_numpy()
    for ax in (ax_v, ax_p, ax_q):
        ax.fill_between(ref.t, 0, 1, where=low, transform=ax.get_xaxis_transform(),
                        color=C_GREY, alpha=0.6, linewidth=0, step="mid", zorder=0)
        ax.grid(color=C_GRID, linewidth=0.5)
        ax.tick_params(labelsize=7)

    # Voltage, with the fitted window shaded
    ax_v.axhspan(MAIN["v_lo"], MAIN["v_hi"], color=C_WIN, alpha=0.08, linewidth=0)
    ax_v.plot(ref.t, ref.V, color=C_V, lw=1.1)
    for v, ls in [(VV["V3"], ":"), (MAIN["v_lo"], "--"), (MAIN["v_hi"], "--")]:
        ax_v.axhline(v, color=C_V, lw=0.7, ls=ls, alpha=0.7)

    # Active power
    ax_p.plot(ref.t, ref.p_pct, color=C_REF, lw=1.1)
    ax_p.plot(est.t, est.p_pct, color=C_EST, lw=1.0, alpha=0.85)
    ax_p.axhline(P_MIN_PCT, color="black", lw=0.7, ls=":")

    # Reactive power vs the required curve (+/- tolerance), using the measured voltage
    req = required_q_pct(ref.V.to_numpy())
    ax_q.fill_between(ref.t, req - TOL_PCT, req + TOL_PCT, color=C_REQ, alpha=0.25, linewidth=0)
    ax_q.plot(ref.t, req, color=C_REQ, lw=1.0)
    ax_q.plot(ref.t, ref.q_pct, color=C_REF, lw=1.1)
    ax_q.plot(est.t, est.q_pct, color=C_EST, lw=1.0, alpha=0.85)
    ax_q.axhline(0, color="black", lw=0.5)

    ax_v.set_title(title, fontsize=9, fontweight="bold")
    day0 = ref.t.dt.normalize().iloc[0]
    ax_q.set_xlim(day0 + pd.Timedelta(hours=DAY_HOURS[0]), day0 + pd.Timedelta(hours=DAY_HOURS[1]))
    ax_q.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax_q.xaxis.set_major_locator(mdates.HourLocator(interval=3))
    ax_q.set_xlabel("Time (AEST)", fontsize=8)
    ax_v.set_ylabel("Voltage (V)", fontsize=8)
    ax_p.set_ylabel("Active power\n(% of rating)", fontsize=8)
    ax_q.set_ylabel("Reactive power (% of rating)\n+ supplying, − absorbing", fontsize=8)

    handles = [
        plt.Line2D([], [], color=C_REF, lw=1.2, label="PV-circuit reference"),
        plt.Line2D([], [], color=C_EST, lw=1.2, label="Unmonitored (net-meter) estimate"),
        Patch(color=C_REQ, alpha=0.4, label=f"Required Q (±{TOL_PCT:.0f}% of rating)"),
        Patch(color=C_WIN, alpha=0.2, label=f"Fitted window ({MAIN['v_lo']}–{MAIN['v_hi']} V)"),
        Patch(color=C_GREY, alpha=0.8, label=f"Output < {P_MIN_PCT:.0f}% of rating (no Q requirement)"),
    ]
    fig.legend(handles=handles, loc="outside lower center", ncol=2, fontsize=7, frameon=False)
    return fig


fig = plot_day_single(title, ref, est)
fname = f"example_day_site{SITE_ID}_{DAY}"
fig.savefig(FIG_DIR / f"{fname}.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / f"{fname}.png", bbox_inches="tight")
plt.show()
print(f"Saved to {FIG_DIR / fname}.pdf/.png")